In [1]:
# REQUIRED LIBRARIES #COPY
import os
import sys
import base64
import pandas as pd
import numpy as np
import textwrap
import requests
import pycountry
import psycopg2
from psycopg2.extras import execute_batch
import json
from tabulate import tabulate
from dotenv import load_dotenv
from pathlib import Path
from cryptography.fernet import Fernet

# GOOGLE AUTH AND SERVICE LIBRARIES
from email.mime.text import MIMEText
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow

# DATABASE CONNECTION
load_dotenv()

# DATABASE CREDENTIALS
DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_PORT = os.getenv("DB_PORT")

IS_CI = os.getenv("GITHUB_ACTIONS", "false").lower() == "true"

SCOPES = ["https://www.googleapis.com/auth/gmail.send"]

# TOKEN ENCRYPTION AND SECURITY SECTION
ENCRYPTION_KEY = os.getenv("TOKEN_ENCRYPTION_KEY")

if not ENCRYPTION_KEY:
    raise ValueError("Missing TOKEN_ENCRYPTION_KEY")

fernet = Fernet(ENCRYPTION_KEY.encode())


def save_token(creds):

    encrypted_token = fernet.encrypt(
        creds.to_json().encode()
    )

# LOCAL BASED ENV AND FILES USE, SECTION
    if not IS_CI:
        with open("token.enc", "wb") as f:
            f.write(encrypted_token)

def load_token():
    try:
        # WEB-BASED ENV USE (GITHUB_ACTIONS), SECTION
        if IS_CI:
            encrypted_token = os.getenv("TOKEN_ENC")

            if not encrypted_token:
                return None

            encrypted_token = encrypted_token.encode()

        else:
            if not os.path.exists("token.enc"):
                return None

            with open("token.enc", "rb") as f:
                encrypted_token = f.read()

        # AUTHENTICATION SECTION FOR BOTH SCENARIOS (LOCAL-BASED / WEB-BASED)
        decrypted_token = fernet.decrypt(
            encrypted_token
        )

        token_info = json.loads(
            decrypted_token.decode()
        )

        return Credentials.from_authorized_user_info(
            token_info,
            SCOPES
        )

    except Exception as e:
        if not IS_CI:
            print(f"Token load failed {e}")
        return None

TO_EMAIL = os.getenv("GMAIL_RECIPIENT")
if not TO_EMAIL:
    raise ValueError("Missing GMAIL_RECIPIENT")

creds = load_token()

# REFRESH IF POSSIBLE WITH 3 ATTEMPTS IN CASE OF FAILURE
if creds and creds.expired and creds.refresh_token:

    refresh_success = False

    for attempt in range(1, 4):

        try:
            creds.refresh(Request())
            save_token(creds)
            refresh_success = True
            break

        except Exception:
            continue

    if not refresh_success:
        creds = None

        if not IS_CI:
            print("Credential refresh attempts failed")

# IF NO CREDS, THEN RUN THE LOGIN POP-UP
if not creds:

    if IS_CI:
        raise RuntimeError("Running in CI environment: interactive OAuth login is not allowed")

    flow = InstalledAppFlow.from_client_secrets_file(
        "gmail_credentials.json",
        SCOPES
    )

    creds = flow.run_local_server(port=0)

# SAVE TOKEN FOR REUSE
if not creds:
    raise RuntimeError("No valid Gmail credentials available")
save_token(creds)

# GMAIL SERVICE INITIALIZATION
service = build("gmail", "v1", credentials=creds)

if not IS_CI:
    print("Gmail API service initialized successfully")

# TESTING THE CONNECTION
cur = None
connection = None

try:
    connection = psycopg2.connect(
        host=DB_HOST,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        port=DB_PORT
    )

    cur = connection.cursor()

    cur.execute("SELECT version();")
    engine_version = cur.fetchone()[0]

    cur.execute("SELECT CURRENT_DATABASE();")
    database_name = cur.fetchone()[0]

    cur.execute("SET search_path TO world_population;")
    cur.execute("SELECT CURRENT_SCHEMA();")
    schema_name = cur.fetchone()[0]

    # EMAIL CONTENT
    subject = "SUCCESSFUL Database Connection - Gmail API"

    body = (
        f"""The database has been successfully connected.
    
Engine version:
{engine_version}

Database and schema name:
{database_name} | {schema_name}
""")

    msg = MIMEText(textwrap.dedent(body))
    msg["to"] = TO_EMAIL
    msg["subject"] = subject

    raw = base64.urlsafe_b64encode(msg.as_bytes()).decode()

    # SEND EMAIL
    service.users().messages().send(
        userId="me",
        body={"raw": raw}
    ).execute()

    if not IS_CI:
        print("Success database connection, email has been sent")

except Exception as e:

    subject = "FAILED Database Connection - Gmail API"

    body = str(e)

    msg = MIMEText(body)
    msg["to"] = TO_EMAIL
    msg["subject"] = subject

    raw = base64.urlsafe_b64encode(msg.as_bytes()).decode()

    service.users().messages().send(
        userId="me",
        body={"raw": raw}
    ).execute()

    if not IS_CI:
        print("Failed database connection, email has been sent")

Credential refresh attempts failed
Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=993643848960-l8merrhu5j385knuhh0qluj38mkvtp77.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A65025%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.send&state=zbhsI6jN1xiIrgqmJiwdH6MiIXfFpU&code_challenge=yMFDA4V0JG_x6jO_2CA1UhnGdxg3bXIaofA_ZDBzkxU&code_challenge_method=S256&access_type=offline
Gmail API service initialized successfully
Success database connection, email has been sent


In [2]:
# WORLD BANK APIs (POPULATION & FERTILITY RATE DATA)

indicator_map = {
    "population": "population_count",
    "fertility": "tfr"
}

apis = {
    "population":"https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL?format=json&per_page=20000",
    "fertility":"https://api.worldbank.org/v2/country/all/indicator/SP.DYN.TFRT.IN?format=json&per_page=20000"
}

results = {}
cleaned_data = {}

api_debug = {}

try:
    for api_name, url in apis.items():
        current_api = api_name

        response = None
        
        for attempt in range(1, 4):
            try:
                response = requests.get(url, timeout=30)
                response.raise_for_status()
                break

            except requests.RequestException:
                if attempt == 3:
                    if not IS_CI:
                        print("API failed, 3 attempts already completed")
                    raise

        api_debug[api_name] = {
            "url": url,
            "status_code": response.status_code,
            "headers": dict(response.headers),
            "response_preview": response.text[:450],
            "response_length": len(response.text)
        }

        try:
            json_data = response.json()
        except ValueError:
            raise ValueError("WORLD BANK API ERROR: Invalid JSON response (likely HTML, empty response, or malformed API request)")

        if not (
            isinstance(json_data, list)
            and len(json_data) > 1
            and isinstance(json_data[1], list)
    ):
            raise ValueError(f"Invalid structure for {api_name}")

        results[api_name] = json_data[1]

except Exception as api_issues:

    subject = "API - ISSUES"

    body = f"""ETL stopped due to API failure.

ERROR MESSAGE:
{api_issues}

FAILED API:
{current_api if 'current_api' in locals() else 'unknown'}

API DEBUG:
{api_debug.get(current_api, {"error": "No debug info available"})}

Response Length:
{api_debug.get(current_api, {}).get("response_length", "unknown")}
"""


    msg = MIMEText(textwrap.dedent(body))
    msg["to"] = TO_EMAIL
    msg["subject"] = subject

    raw = base64.urlsafe_b64encode(msg.as_bytes()).decode()

    service.users().messages().send(
        userId="me",
        body={"raw": raw}
    ).execute()

    if not IS_CI:
        print("API ISSUES")
    sys.exit("ETL STOPPED")

# DATA CLEANING & TRANSFORMATION
def clean_world_bank_data(data, column_name):

    df = pd.json_normalize(data)

    df = df[["country.value", "countryiso3code", "date", "value"]]

    df = df.rename(columns={
    "country.value": "country_name",
    "countryiso3code": "country_code",
    "date": "year_record",
    "value": column_name
})

    inclusion = {"XKX", "CHI"}
    valid_country_codes = {country.alpha_3 for country in pycountry.countries}
    valid_country_codes = valid_country_codes.union(inclusion)

    df = df[df["country_code"].isin(valid_country_codes)]

    df["country_name"] = df["country_name"].astype("string").str.strip()
    df["country_code"] = df["country_code"].str.strip().str.upper()

    df["year_record"] = pd.to_numeric(df["year_record"], errors="coerce").astype("Int64")
    df[column_name] = pd.to_numeric(df[column_name], errors="coerce")

    return df

# CLEANING FOR BOTH APIs DATA REUSING THE FUNCTION "clean_world_bank_data".
for api_name, data in results.items():

    column_name = indicator_map[api_name]

    cleaned_data[api_name] = clean_world_bank_data(data, column_name)

population_df = cleaned_data["population"]
fertility_df = cleaned_data["fertility"]

country_names = {
"Venezuela, RB": "Venezuela",
"Iran, Islamic Rep.": "Iran",
"Korea, Rep.": "South Korea",
"Korea, Dem. People's Rep.": "North Korea",
"Egypt, Arab Rep.": "Egypt",
"Russian Federation": "Russia",
"Syrian Arab Republic": "Syria",
"Yemen, Rep.": "Yemen",
"Viet Nam": "Vietnam",
"Tanzania, United Republic of": "Tanzania",
"St. Martin (French part)": "French St. Martin",
"Sint Maarten (Dutch part)": "Dutch Sint Maarten",
"Puerto Rico (US)": "Puerto Rico",
"Micronesia, Fed. Sts.": "Micronesia",
"Moldova, Republic of": "Moldova",
"Bahamas, The": "Bahamas",
"Virgin Islands, British": "British Virgin Islands",
"Virgin Islands (U.S.)": "American Virgin Islands",
"West Bank and Gaza": "Palestine",
"Congo, Dem. Rep.": "Democratic Republic of the Congo",
"Congo, Rep.": "Republic of the Congo",
"Somalia, Fed. Rep.": "Somalia",
"Slovak Republic": "Republic of Slovakia",
"Gambia, The": "Gambia"
}

def rename_countries(country):
    if pd.isna(country):
        return country
    country = str(country).strip()
    return country_names.get(country, country)


population_df["country_name"] = population_df["country_name"].apply(rename_countries)
fertility_df["country_name"] = fertility_df["country_name"].apply(rename_countries)

#DATA ENRICHEMENT

def get_project_root():

    # IF CI ENVS
    env_root = os.getenv("PROJECT_ROOT")
    if env_root:
        return Path(env_root)

    # IF SCRIPT FILE
    if "__file__" in globals():
        return Path(__file__).resolve().parent

    # IF NOTEBOOK FALLBACK
    return Path.cwd()

BASE_DIR = get_project_root()

file_path = BASE_DIR / "cleaned_datasets" / "country_enrichment.csv"

if not file_path.exists():
    msg = f"Missing file at: {file_path}"

    if not IS_CI:
        print(msg)
        
    raise FileNotFoundError(msg)

country_enrichment = pd.read_csv(file_path)
enrichment_details = country_enrichment[["country_code", "capital", "land_area_km2", "continent"]]
enrichment_details["country_code"] = enrichment_details["country_code"].str.strip().str.upper()

try:
    population_df = population_df.merge(enrichment_details, on=["country_code"], how="left")
    population_df = population_df[["country_name", "country_code", "year_record", "population_count", "land_area_km2", "capital", "continent"]]
    fertility_df = fertility_df[["country_name", "country_code", "year_record", "tfr"]]
except Exception as mergeissue:
    if not IS_CI:
        print(mergeissue) 
    subject = "Countries_Enrichment - Gmail API"

    body = f"""The enrichmenet has failed due to.
    
Error:
{mergeissue}
"""

    msg = MIMEText(textwrap.dedent(body))
    msg["to"] = TO_EMAIL
    msg["subject"] = subject

    raw = base64.urlsafe_b64encode(msg.as_bytes()).decode()

    # SEND EMAIL
    service.users().messages().send(
        userId="me",
        body={"raw": raw}
    ).execute()


#FINAL DATA CLEANING
population_df["capital"] = population_df["capital"].astype("string").str.strip().str.title()
population_df["continent"] = population_df["continent"].astype("string").str.strip().str.title()
population_df["land_area_km2"] = pd.to_numeric(population_df["land_area_km2"], errors="coerce")

population_df["population_count"] = pd.to_numeric(population_df["population_count"], errors="coerce").astype("Int64")
fertility_df["tfr"] = pd.to_numeric(fertility_df["tfr"], errors="coerce")

#QUICK DATA VALIDATION
if not IS_CI:
    print("POPULATION DATA")
    print(population_df.head(10))

    print("\nFERTILITY DATA:")
    print(fertility_df.head(5))

POPULATION DATA
  country_name country_code  year_record  population_count  land_area_km2  \
0  Afghanistan          AFG         2025              <NA>         652230   
1  Afghanistan          AFG         2024          42647492         652230   
2  Afghanistan          AFG         2023          41454761         652230   
3  Afghanistan          AFG         2022          40578842         652230   
4  Afghanistan          AFG         2021          40000412         652230   
5  Afghanistan          AFG         2020          39068979         652230   
6  Afghanistan          AFG         2019          37856121         652230   
7  Afghanistan          AFG         2018          36743039         652230   
8  Afghanistan          AFG         2017          35688935         652230   
9  Afghanistan          AFG         2016          34700612         652230   

  capital continent  
0   Kabul      Asia  
1   Kabul      Asia  
2   Kabul      Asia  
3   Kabul      Asia  
4   Kabul      Asia  
5   

In [3]:
#PREPARING DATA FOR DIM_YEAR VALUES INSERTION

# PANDAS NAN VALUES HANDLING
def to_pg_value(x):
    if pd.isna(x):
        return None
    if isinstance(x, np.generic):
        return x.item()
    return x


def df_to_pg_records(df):
    return [
        tuple(to_pg_value(x) for x in row)
        for row in df.itertuples(index=False, name=None)
    ]

dim_year_table = (population_df[["year_record"]].drop_duplicates().dropna(subset=["year_record"]).assign(year_record=lambda df:pd.to_numeric(df["year_record"], errors="raise")))
dim_year_table_values = df_to_pg_records(dim_year_table)

dim_year_insert_query ="""
INSERT INTO dim_year (year_record)
VALUES (%s)
ON CONFLICT (year_record) DO NOTHING;
"""

dim_countries_table = population_df[["country_name", "country_code", "capital", "continent", "land_area_km2"]].drop_duplicates().dropna(subset=["country_code"])
dim_countries_table_values = df_to_pg_records(dim_countries_table)

dim_countries_insert_query = """
INSERT INTO dim_countries (country_name, country_code, capital, continent, land_area_km2)
VALUES (%s, %s, %s, %s, %s)
ON CONFLICT (country_code)
DO UPDATE SET
    country_name = EXCLUDED.country_name,
    capital = EXCLUDED.capital,
    continent = EXCLUDED.continent,
    land_area_km2 = EXCLUDED.land_area_km2
WHERE
    dim_countries.country_name IS DISTINCT FROM EXCLUDED.country_name
    OR dim_countries.capital IS DISTINCT FROM EXCLUDED.capital
    OR dim_countries.continent IS DISTINCT FROM EXCLUDED.continent
    OR dim_countries.land_area_km2 IS DISTINCT FROM EXCLUDED.land_area_km2;
    """

fact_population_table = population_df[["country_code", "year_record", "population_count"]].drop_duplicates(subset=["country_code", "year_record"]).dropna(subset=["year_record", "country_code"])
fact_population_table_values = df_to_pg_records(fact_population_table)

fact_population_insert_query ="""
INSERT INTO fact_population
(country_code, year_record, population_count)
VALUES (%s, %s, %s)

ON CONFLICT (country_code, year_record)

DO UPDATE
SET population_count = EXCLUDED.population_count

WHERE fact_population.population_count
IS DISTINCT FROM EXCLUDED.population_count;
"""

fact_fertility_rate_table = fertility_df[["country_code", "year_record", "tfr"]].drop_duplicates(subset=["country_code", "year_record"]).dropna(subset=["year_record", "country_code"])
fact_fertility_rate_table_values = df_to_pg_records(fact_fertility_rate_table)

fact_fertility_rate_insert_query ="""
INSERT INTO fact_fertility_rate
(country_code, year_record, tfr)
VALUES (%s, %s, %s)

ON CONFLICT (country_code, year_record)

DO UPDATE
SET tfr = EXCLUDED.tfr

WHERE fact_fertility_rate.tfr
IS DISTINCT FROM EXCLUDED.tfr;"""

In [4]:
# EMAIL HEADERS
headers = ["Country", "Code", "Total Years Recorded"]

try:
   # LOAD DIM_YEAR
    execute_batch(cur,dim_year_insert_query, dim_year_table_values, page_size=1000)

    # LOAD DIM_COUNTRIES
    execute_batch(cur, dim_countries_insert_query, dim_countries_table_values, page_size=1000)
    
    #LOAD FACT_POPULATION 
    execute_batch(cur, fact_population_insert_query, fact_population_table_values, page_size=1000)

    #LOAD FACT_FERILTITY_RATE
    execute_batch(cur, fact_fertility_rate_insert_query, fact_fertility_rate_table_values, page_size=1000)

    #COMMIT THE INSERTION
    connection.commit()

    # DIM_YEAR QUERY
    cur.execute("SELECT COUNT(*) AS years_count FROM dim_year")
    dim_year_query = cur.fetchone()[0]
    cur.execute("SELECT MIN(year_record) FROM dim_year")
    min_dim_year = cur.fetchone()[0]
    cur.execute("SELECT MAX(year_record) FROM dim_year")
    max_dim_year = cur.fetchone()[0]

    # DIM_COUNTRIES QUERY
    cur.execute("SELECT COUNT(country_code) AS total_countries FROM dim_countries")
    dim_countries_query = cur.fetchone()[0]

    # FACT_POPULATION QUERY
    cur.execute("""SELECT
                dc.country_name AS Countries,
                dc.country_code AS Code,
                COUNT(fp.year_record) AS total_years_recorded
                FROM dim_countries AS dc
                LEFT JOIN fact_population AS fp
                ON dc.country_code = fp.country_code 
                GROUP BY dc.country_name, dc.country_code
                ORDER BY total_years_recorded DESC
                """)

    # EMAIL PRETTY PRINT :)
    fact_population_query = cur.fetchall()
    population_summary = tabulate(fact_population_query, headers=headers, tablefmt="grid")

    # FACT_FERTILITY_RATE QUERY
    cur.execute("""SELECT
                dc.country_name AS Countries,
                dc.country_code AS Code,
                COUNT(ff.year_record) AS total_years_recorded
                FROM dim_countries AS dc
                LEFT JOIN fact_fertility_rate AS ff
                ON dc.country_code = ff.country_code 
                GROUP BY dc.country_name, dc.country_code
                ORDER BY total_years_recorded DESC
                """)

    # EMAIL PRETTY PRINT :)
    fact_fertility_query = cur.fetchall()
    fertility_summary = tabulate(fact_fertility_query, headers=headers, tablefmt="grid")

    # EMAIL NOTIFICATION OF THE SUCCESSFUL INSERTION WiTH THE CONFIRMATION OF THE NUMBER OF VALUES IN THE TABLE
    subject = "SUCCESSFUL DATABASE INSERTION"
    body = f"""
    
The YEAR table has been successfully updated.
Total years count: {dim_year_query}
MIN year: {min_dim_year}
MAX year: {max_dim_year}

The COUNTRIES table has been successfully updated.
Total countries count: {dim_countries_query}

The POPULATION table has been successfully updated.
{population_summary}

The FERTILITY table has been successfully updated.
{fertility_summary}
"""

    msg = MIMEText(textwrap.dedent(body))
    msg["to"] = TO_EMAIL
    msg["subject"] = subject

    raw = base64.urlsafe_b64encode(msg.as_bytes()).decode()
    service.users().messages().send(
    userId="me",
    body={"raw": raw}
    ).execute()

    if not IS_CI:
        print("SUCCESSFUL DATABASE INSERTION. EMAIL HAS BEEN SENT")

except Exception as e:
    # ROLLBACK TRANSACTION AND SEND FAILURE NOTIFICATION
    connection.rollback()

    subject = "FAILED VALUES INSERTION"
    body = (
    f"""Failed insertions due to error message:

{e}"""
    )

    msg = MIMEText(textwrap.dedent(body))
    msg["to"] = TO_EMAIL
    msg["subject"] = subject

    raw = base64.urlsafe_b64encode(msg.as_bytes()).decode()
    service.users().messages().send(
    userId="me",
    body={"raw": raw}
    ).execute()

    if not IS_CI:
        print(f"""Error occurred: {e}. email sent""")
    raise e
    
    # AFTER THE SCRIPT RUNS SUCCESSFULLY, CLOSE THE DB CONNECTION
finally:

    if cur is not None:
        cur.close()

    if connection is not None:
        connection.close()

SUCCESSFUL DATABASE INSERTION. EMAIL HAS BEEN SENT
